In [ ]:
# !pip uninstall -y hf_xet

In [ ]:
!wget "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [ ]:
# !pip install -U "huggingface_hub[hf_xet]"

In [2]:
import os
import shutil
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import random

load_dotenv()

# 로컬
# ROOT = Path(os.environ["DATA_ROOT"])
# HF_HOME = ROOT / ".hf_cache"
# os.environ["HF_HOME"] = str(HF_HOME)

# 클라우드
from google.colab import userdata
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "patent_disc"
os.environ["HF_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
os.environ["HF_HOME"] = ".hf_cache"
# os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import load_dataset, load_from_disk
from sklearn.metrics import f1_score

In [3]:
# Config
config = {
    "num_labels": 188,
    "seed": 42,
    "learning_rate": 3e-5,
    "epochs": 12,
    "early_stop": 6,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "model_name": "skt/A.X-Encoder-base",
    "max_len": 8192,          # >8,192 극소수(<1%)만 절단 → 사실상 full (코퍼스 max 10,523)
    "eff_batch": 8,           # max_len=512 런과 유효 배치 등화 → 컨텍스트 길이 외 변수 고정
    "micro_batch": 4,
    "eval_micro_batch": 4,
    "repo_train": "ingyoun/A.X-patent-maxlen8192-train",
    "repo_final": "ingyoun/A.X-patent-maxlen8192",
    "run_name": "modernbert_maxlen=8192_v2",
    "out_path": "/content/output/modernbert-maxlen8192",
    "tag": "modernbert-patent-len8192",
}
config["grad_accum"] = config["eff_batch"] // config["micro_batch"]   # micro×accum = eff_batch

In [4]:
random.seed(config['seed'])
np.random.seed(config['seed'])
torch.manual_seed(config['seed'])
torch.cuda.manual_seed_all(config['seed'])

In [5]:
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

cuda


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 데이터셋

In [7]:
dataset = load_dataset(
    "ingyoun/patent-clean-text-modernbert-tokenized",
    cache_dir="/content/drive/MyDrive/.hf_cache",
)

dataset

README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 11162
    })
})

## 모델

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
        pretrained_model_name_or_path=config["model_name"],
        num_labels=config["num_labels"],
        problem_type="multi_label_classification",
        classifier_dropout=0.5,
        dtype=torch.float32,
        attn_implementation="flash_attention_2",   # 장문처리 전략
    )

## 토크나이저

In [9]:
REV = "9708f9c404ace91efd25c06fac2d73413616f4ef"
tokenizer = AutoTokenizer.from_pretrained(config["model_name"], revision=REV)
EOS_ID = tokenizer.eos_token_id


def _prep(batch):
    max_len = config["max_len"]
    ids, masks = [], []
    for x, m in zip(batch["input_ids"], batch["attention_mask"]):
        if len(x) > max_len:
            x = x[: max_len - 1] + [EOS_ID]   # <s> 유지 + 꼬리를 <\s>로 마감
            m = m[:max_len]
        ids.append(x)
        masks.append(m)
    return {"input_ids": ids, "attention_mask": masks, "length": [len(i) for i in ids]}


dataset = dataset.map(_prep, batched=True)
print(f"EOS_ID={EOS_ID}  max_len={config['max_len']}")
dataset

tokenizer_config.json:   0%|          | 0.00/6.95k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

EOS_ID=1  max_len=8192


DatasetDict({
    train: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 201895
    })
    test: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11271
    })
    val: Dataset({
        features: ['document_id', 'kobert_len', 'length_bin', 'input_ids', 'attention_mask', 'labels', 'length'],
        num_rows: 11162
    })
})

## 커스텀

In [10]:
class FocalLoss(nn.Module):
    def __init__(self, alpha: float = 0.25, gamma: int = 2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

In [11]:
class FocalTrainer(Trainer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k)
        self.focal = FocalLoss(0.25, 2.0)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        loss = self.focal(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss

In [12]:
class MultiLabelCollator:
    def __init__(self, tokenizer):
        self.tok = tokenizer

    def __call__(self, feats):
        labels = torch.tensor([f["labels"] for f in feats], dtype=torch.float)
        keys = ("input_ids", "attention_mask")
        enc = [{k: f[k] for k in keys if k in f} for f in feats]
        batch = self.tok.pad(enc, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch

In [13]:
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(eval_pred):
    """03_02_Metric의 headline과 동일 정의 — sigmoid→τ=0.5 멀티라벨 F1(micro/macro/sample).
    micro_f1이 모델 선택(metric_for_best_model) 기준. 벤더 연속성 앵커(top-1 weighted)도 병기.
    """
    logits, labels = eval_pred
    logits = np.asarray(logits)
    Y = np.asarray(labels).astype(int)
    pred = (_sigmoid(logits) >= 0.5).astype(int)
    return {
        "micro_f1":  f1_score(Y, pred, average="micro",   zero_division=0),   # headline (모델 선택 기준)
        "macro_f1":  f1_score(Y, pred, average="macro",   zero_division=0),
        "sample_f1": f1_score(Y, pred, average="samples", zero_division=0),
        "empty_rate": float((pred.sum(1) == 0).mean()),
        "anchor_weighted_f1": f1_score(Y.argmax(1), logits.argmax(1), average="weighted", zero_division=0),  # 벤더 연속성
    }

## 훈련

In [14]:
training_args = TrainingArguments(
    output_dir='/content/results',
    seed=config["seed"],
    learning_rate=config["learning_rate"],
    weight_decay=config["weight_decay"],
    lr_scheduler_type="linear",
    warmup_ratio=config["warmup_ratio"],
    per_device_train_batch_size=config["micro_batch"],
    per_device_eval_batch_size=config["eval_micro_batch"],
    gradient_accumulation_steps=config["grad_accum"],   # micro×accum = eff_batch (max_len=512 런과 등화)
    train_sampling_strategy="group_by_length",          # 유사 길이 배치로 padding 최소화
    remove_unused_columns=False,                        # 커스텀 collator가 키를 직접 선택 + length 컬럼 보존
    num_train_epochs=config["epochs"],
    bf16=True,                                          # ModernBERT 계열 안정성엔 bf16 (bf16=True, fp16=False)
    eval_strategy='steps',
    eval_steps=12618,
    save_strategy='steps',
    save_steps=12618,
    save_total_limit=3,
    logging_dir='/content/logs',
    logging_steps=50,
    metric_for_best_model="micro_f1",                   # 03_02 headline 기준 모델 선택
    greater_is_better=True,
    load_best_model_at_end=True,
    push_to_hub=True,
    hub_model_id=config["repo_train"],
    hub_strategy="checkpoint",
    report_to="wandb",
    run_name=config["run_name"]
)                                                       # optim="adamw_torch_fused" -> 옵티마이저 state 트래픽 감소 -> fp32 오버헤드 일부 상쇄

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=MultiLabelCollator(tokenizer),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stop"])]
)

In [ ]:
# ## micro batch 확인 — worst-case 배치로 실제 스텝을 돌려 peak 측정
# def probe_batches(model, vocab_size, seq_len, train_mb=(1, 2, 4), eval_mb=(2, 4, 8), num_labels=188):
#     """worst case = 모든 시퀀스가 seq_len이고 mask가 전부 1(패딩 0).

#     group_by_length가 all-long 배치를 반드시 만들므로 이 경우는 확실히 발생한다
#     (random 샘플링이어도 배치에 장문 하나만 섞이면 배치 전체가 그 길이로 패딩된다).

#     - AdamW state(exp_avg/exp_avg_sq ≈ params×8B)를 상주시킨 **정상상태** peak를 잰다.
#       lr=0이라 가중치는 변하지 않는다 → 사전학습 가중치를 훼손하지 않음.
#     - eval도 잰다: forward-only(no_grad)라 싸지만 eval 배치 역시 배치 내 최댓값으로 패딩되고,
#       eval OOM은 첫 에폭 끝에서 런을 죽인다.
#     """
#     assert model.device.type == "cuda", "모델이 GPU에 없습니다. — Trainer 생성 후 실행할 것"
#     dev = model.device
#     opt = torch.optim.AdamW(model.parameters(), lr=0.0)   # lr=0 → state만 할당, 가중치 불변

#     def _mk(mb):
#         ids = torch.randint(5, vocab_size, (mb, seq_len), device=dev)
#         return ids, torch.ones_like(ids), torch.zeros(mb, num_labels, device=dev)

#     model.train()
#     for mb in train_mb:
#         try:
#             ids, mask, labels = _mk(mb)
#             for step in (0, 1):                       # step0: AdamW state 할당 / step1: 정상상태 peak 측정
#                 if step == 1:
#                     torch.cuda.reset_peak_memory_stats()
#                 with torch.autocast("cuda", dtype=torch.bfloat16):   # Trainer(bf16=True)와 동일 경로
#                     out = model(input_ids=ids, attention_mask=mask)
#                     loss = F.binary_cross_entropy_with_logits(out.logits.float(), labels)
#                 loss.backward()                                      # backward는 autocast 밖
#                 opt.step()
#                 opt.zero_grad(set_to_none=True)
#             print(f"train micro={mb:>2}: peak {torch.cuda.max_memory_allocated()/1e9:5.1f} GB  OK")
#         except torch.cuda.OutOfMemoryError:
#             print(f"train micro={mb:>2}: OOM")
#             opt.zero_grad(set_to_none=True)
#             break                                     # 이후 후보는 자명하게 OOM
#         finally:
#             torch.cuda.empty_cache()

#     model.eval()
#     for mb in eval_mb:                                # 옵티마이저 state가 상주한 실제 조건에서 측정
#         try:
#             torch.cuda.reset_peak_memory_stats()
#             ids, mask, _ = _mk(mb)
#             with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
#                 model(input_ids=ids, attention_mask=mask)
#             print(f"eval  micro={mb:>2}: peak {torch.cuda.max_memory_allocated()/1e9:5.1f} GB  OK")
#         except torch.cuda.OutOfMemoryError:
#             print(f"eval  micro={mb:>2}: OOM")
#             break
#         finally:
#             torch.cuda.empty_cache()

#     del opt                                           # Trainer가 자체 옵티마이저를 새로 만들도록 정리
#     model.zero_grad(set_to_none=True)
#     model.train()
#     torch.cuda.empty_cache()
#     torch.cuda.reset_peak_memory_stats()              # train() peak를 깨끗하게 재측정하기 위함


# # 설정값 검증. 여유를 보려면 train_mb=(1, 2, 4), eval_mb=(2, 4, 8)로 넓혀 실행
# probe_batches(
#     model, tokenizer.vocab_size, config["max_len"],
#     train_mb=(1,2,4,8,),
#     eval_mb=(4,8,),
#     num_labels=config["num_labels"],
# )

train micro= 1: peak   7.2 GB  OK
train micro= 2: peak  12.3 GB  OK
train micro= 4: peak  22.4 GB  OK
train micro= 8: OOM
eval  micro= 4: peak   3.0 GB  OK
eval  micro= 8: peak   3.9 GB  OK


In [16]:
import gc

# probe_batches가 옵티마이저·캐시를 내부에서 정리하므로 여기선 GC만 한 번 더 돌린다
gc.collect()
torch.cuda.empty_cache()
print(f"probe 후 잔여 allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB (모델 가중치)")

probe 후 잔여 allocated: 0.6 GB (모델 가중치)


In [ ]:
# trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
12618,0.002623,0.000862,0.658684,0.627972,0.600997,0.231679,0.662509
25236,0.001971,0.000697,0.731230,0.713948,0.713501,0.111270,0.721959
37854,0.001591,0.000583,0.758632,0.740806,0.731297,0.134922,0.754227


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [17]:
from huggingface_hub import snapshot_download

def copy_checkpoint(repo_train, dst):
    src = snapshot_download(repo_id=repo_train, allow_patterns="last-checkpoint/*")
    shutil.copytree(os.path.join(src, "last-checkpoint"), dst, dirs_exist_ok=True)

In [ ]:
# copy_checkpoint(config["repo_train"], "/content/results/checkpoint-37854")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

'/content/results/checkpoint-37854'

In [ ]:
# trainer.train(resume_from_checkpoint="/content/results/checkpoint-37854")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
50472,0.001002,0.000484,0.801052,0.793103,0.801815,0.055814,0.777512
63090,0.000897,0.000441,0.817858,0.810477,0.819548,0.050081,0.788206
75708,0.001044,0.000408,0.828453,0.824290,0.835569,0.036015,0.796691
88326,0.000566,0.000417,0.836259,0.828819,0.845529,0.031804,0.805550
100944,0.000577,0.000391,0.843817,0.838388,0.853481,0.026787,0.808574
113562,0.000372,0.000446,0.843491,0.838706,0.851684,0.031177,0.809837


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# dst = "/content/results/checkpoint-113562"
# copy_checkpoint(config["repo_train"], dst)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
# trainer.train(resume_from_checkpoint=dst)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
126180,0.000331,0.000441,0.851801,0.849073,0.865733,0.018008,0.813306
138798,0.000322,0.000452,0.852474,0.848788,0.864962,0.018903,0.816239
151416,0.000392,0.000428,0.855193,0.851779,0.864506,0.024548,0.816509
164034,0.000226,0.000511,0.854949,0.851256,0.868971,0.017022,0.815462


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [18]:
dst = "/content/results/checkpoint-214506"
copy_checkpoint(config["repo_train"], dst)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [19]:
trainer.train(resume_from_checkpoint=dst)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: paraise (paraise-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
227124,0.000097,0.000725,0.862368,0.859785,0.878955,0.011378,0.817504
239742,0.000075,0.000715,0.865922,0.863312,0.881003,0.013528,0.821206
252360,0.000055,0.000729,0.867277,0.864307,0.880994,0.014782,0.823483
264978,0.000029,0.000852,0.867465,0.865082,0.883449,0.012095,0.822723
277596,0.000050,0.000867,0.866870,0.864215,0.883145,0.010930,0.822967
290214,0.000008,0.000957,0.868942,0.866399,0.884239,0.012543,0.823380
302832,0.000013,0.000983,0.869969,0.867471,0.884878,0.012543,0.823577
302844,0.000013,0.000983,0.870011,0.867487,0.884866,0.012543,0.823422


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=302844, training_loss=1.3467702142567089e-05, metrics={'train_runtime': 33291.1546, 'train_samples_per_second': 72.774, 'train_steps_per_second': 9.097, 'total_flos': 1.329597953834969e+18, 'train_loss': 1.3467702142567089e-05, 'epoch': 12.0})

## 평가

In [20]:
test_metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
for k, v in test_metrics.items():
    print(f"{k}: {v}")

[transformers] early stopping required metric_for_best_model, but did not find eval_micro_f1 so early stopping is disabled


Training Loss,Validation Loss,Step,Micro F1,Macro F1,Sample F1,Empty Rate,Anchor Weighted F1
0.000013,0.000988,302844,0.868375,0.864756,0.882527,0.013397,0.825683


test_loss: 0.0009883588645607233
test_micro_f1: 0.8683745583038869
test_macro_f1: 0.8647564273657622
test_sample_f1: 0.8825266453693391
test_empty_rate: 0.013397214089255611
test_anchor_weighted_f1: 0.825683068416136


In [21]:
os.makedirs(config["out_path"], exist_ok=True)
trainer.save_model(config["out_path"])
tokenizer.save_pretrained(config["out_path"])

metrics_fp = os.path.join(config["out_path"], f"{config['tag']}_test_metrics.json")
with open(metrics_fp, "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)

print("saved", config["out_path"])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Invalid model-index. Not loading eval results into CardData.


Upload 0 LFS files: 0it [00:00, ?it/s]

saved /content/output/modernbert-maxlen8192


In [23]:
shutil.copy(os.path.join(config["out_path"], f"{config['tag']}_test_metrics.json"), "/content/drive/MyDrive/patent_disc/")

'/content/drive/MyDrive/patent_disc/modernbert-patent-len8192_test_metrics.json'

In [25]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

/content/results/checkpoint-302844
0.8700107674599933


In [27]:
trainer.save_model()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Invalid model-index. Not loading eval results into CardData.


Upload 0 LFS files: 0it [00:00, ?it/s]

No files have been modified since last commit. Skipping to prevent empty commit.


In [29]:
trainer.model.push_to_hub(config["repo_final"])
tokenizer.push_to_hub(config["repo_final"])

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/598M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/ingyoun/A.X-patent-maxlen8192/commit/c0a41b48344a576fb9e48af520d520bea000dfdf', commit_message='Upload tokenizer', commit_description='', oid='c0a41b48344a576fb9e48af520d520bea000dfdf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ingyoun/A.X-patent-maxlen8192', endpoint='https://huggingface.co', repo_type='model', repo_id='ingyoun/A.X-patent-maxlen8192'), pr_revision=None, pr_num=None)